In [ ]:
# Importing Libraries
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Uploading the CSV
from google.colab import files
files.upload()

Saving HKRE0_Nairobi_-_Doonholm_cleaned.csv to HKRE0_Nairobi_-_Doonholm_cleaned.csv


{'HKRE0_Nairobi_-_Doonholm_cleaned.csv': b'time,tavg,tmin,tmax,prcp,wspd,pres\n2015-01-27,21.2,14.0,29.0,0.7,11.8,1019.3\n2015-01-28,21.6,14.0,29.0,0.7,15.1,1019.3\n2015-01-29,21.6,15.0,28.0,0.7,12.9,1019.3\n2015-01-30,22.1,15.0,28.0,0.7,15.5,1019.3\n2015-02-01,22.5,15.0,28.0,0.7,15.4,1019.3\n2015-02-02,22.2,15.0,28.0,0.7,13.6,1019.3\n2015-02-03,22.2,15.0,28.0,0.7,16.0,1019.3\n2015-02-04,21.4,15.0,28.0,0.7,19.5,1019.3\n2015-02-05,22.0,15.0,28.0,0.7,16.8,1019.3\n2015-02-07,22.3,15.0,28.0,0.7,16.1,1019.3\n2015-02-08,21.3,15.0,28.0,0.7,15.3,1019.3\n2015-02-09,22.0,15.0,28.0,0.7,17.0,1019.3\n2015-02-10,21.8,16.0,28.0,0.7,22.3,1019.3\n2015-02-11,22.2,16.0,28.0,0.7,12.9,1019.3\n2015-02-12,22.7,16.0,28.0,0.7,11.3,1019.3\n2015-02-13,23.6,17.0,30.0,0.7,12.6,1019.3\n2015-02-14,23.1,17.0,30.0,0.7,11.6,1019.3\n2015-02-15,21.0,19.0,24.0,0.7,11.6,1019.3\n2015-02-16,21.6,18.0,28.0,0.7,7.5,1019.3\n2015-02-17,20.6,18.0,28.0,0.7,14.1,1019.3\n2015-02-18,22.1,18.0,27.0,0.7,11.9,1019.3\n2015-02-19,22.0,17.

In [ ]:
df = pd.read_csv("HKRE0_Nairobi_-_Doonholm_cleaned.csv")

df.head()

,time,tavg,tmin,tmax,prcp,wspd,pres
0,2015-01-27,21.2,14.0,29.0,0.7,11.8,1019.3
1,2015-01-28,21.6,14.0,29.0,0.7,15.1,1019.3
2,2015-01-29,21.6,15.0,28.0,0.7,12.9,1019.3
3,2015-01-30,22.1,15.0,28.0,0.7,15.5,1019.3
4,2015-02-01,22.5,15.0,28.0,0.7,15.4,1019.3


In [ ]:
# Parse dates and extract months
df["time"] = pd.to_datetime(df["time"])
df["month"] = df["time"].dt.month

In [ ]:
# Define a reusable monthly-stat function
def compute_monthly_components(df, column, threshold=None):
    grouped = df.groupby("month")

    avg = grouped[column].mean()
    variability = grouped[column].std() / avg  # coefficient of variation

    if threshold is not None:
        frequency = grouped.apply(lambda x: (x[column] > threshold).mean())
        intensity = grouped.apply(
            lambda x: x[x[column] > threshold][column].mean()
        )
    else:
        frequency = grouped[column].apply(lambda x: (x > 0).mean())
        intensity = grouped[column].mean()

    return pd.DataFrame({
        "avg": avg,
        "frequency": frequency,
        "intensity": intensity,
        "variability": variability
    })

In [ ]:
# Compute rainfall components
rain_stats = compute_monthly_components(df, "prcp")
rain_stats

,avg,frequency,intensity,variability
month,,,,
1,0.700000,1.000000,0.700000,0.000000
2,0.700000,1.000000,0.700000,0.000000
3,0.700000,1.000000,0.700000,0.000000
4,0.700000,1.000000,0.700000,0.000000
5,0.700000,1.000000,0.700000,0.000000
6,0.700000,1.000000,0.700000,0.000000
7,0.305556,0.444444,0.305556,1.491209
8,1.989655,0.655172,1.989655,1.900709
9,1.176190,0.619048,1.176190,1.995877


In [ ]:
# Normalize rainfall components (0–1 scale)
rain_norm = (rain_stats - rain_stats.min()) / (rain_stats.max() - rain_stats.min())

In [ ]:
# Compute rainfall index
rain_stats["rain_index"] = rain_norm.mean(axis=1)

rain_index = rain_stats[["rain_index"]].reset_index()
rain_index

,month,rain_index
0,1,0.303533
1,2,0.303533
2,3,0.303533
3,4,0.303533
4,5,0.303533
5,6,0.303533
6,7,0.186786
7,8,0.561470
8,9,0.446733
9,10,0.844605


In [ ]:
# Temperature index
tavg_threshold = df["tavg"].quantile(0.75)

temp_stats = compute_monthly_components(
    df, "tavg", threshold=tavg_threshold
)

/tmp/ipython-input-650243391.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frequency = grouped.apply(lambda x: (x[column] > threshold).mean())
/tmp/ipython-input-650243391.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = grouped.apply(


In [ ]:
# Normalize & Compute temperature index
temp_norm = (temp_stats - temp_stats.min()) / (temp_stats.max() - temp_stats.min())
temp_stats["temp_index"] = temp_norm.mean(axis=1)

temp_index = temp_stats[["temp_index"]].reset_index()

In [ ]:
# Wind speed index
wind_threshold = df["wspd"].quantile(0.75)

wind_stats = compute_monthly_components(
    df, "wspd", threshold=wind_threshold
)

/tmp/ipython-input-650243391.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frequency = grouped.apply(lambda x: (x[column] > threshold).mean())
/tmp/ipython-input-650243391.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = grouped.apply(


In [ ]:
# Normalize & Compute wind speed
wind_norm = (wind_stats - wind_stats.min()) / (wind_stats.max() - wind_stats.min())
wind_stats["wind_index"] = wind_norm.mean(axis=1)

wind_index = wind_stats[["wind_index"]].reset_index()

In [ ]:
# Pressure index
pres_threshold = df["pres"].quantile(0.25)

pres_stats = compute_monthly_components(
    df, "pres", threshold=pres_threshold
)

/tmp/ipython-input-650243391.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  frequency = grouped.apply(lambda x: (x[column] > threshold).mean())
/tmp/ipython-input-650243391.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = grouped.apply(


In [ ]:
# Normalize & Invert
pres_norm = (pres_stats - pres_stats.min()) / (pres_stats.max() - pres_stats.min())

pres_stats["pressure_index"] = 1 - pres_norm.mean(axis=1)

pressure_index = pres_stats[["pressure_index"]].reset_index()

In [ ]:
# Final Monthly Climate Index
# Merge all indices
monthly_index = rain_index \
    .merge(temp_index, on="month") \
    .merge(wind_index, on="month") \
    .merge(pressure_index, on="month")

In [ ]:
# Compute final climate index
monthly_index["climate_index"] = monthly_index[
    ["rain_index", "temp_index", "wind_index", "pressure_index"]
].mean(axis=1)

monthly_index

,month,rain_index,temp_index,wind_index,pressure_index,climate_index
0,1,0.303533,0.430559,0.478994,0.250000,0.365771
1,2,0.303533,0.841316,0.762508,0.250000,0.539339
2,3,0.303533,0.661083,0.794409,0.250000,0.502256
3,4,0.303533,0.668333,0.390015,0.250000,0.402970
4,5,0.303533,0.173993,0.369305,0.250000,0.274208
5,6,0.303533,0.412781,0.162698,0.250000,0.282253
6,7,0.186786,0.242817,0.144158,0.129421,0.175796
7,8,0.561470,0.258361,0.119847,0.387967,0.331911
8,9,0.446733,0.254134,0.227661,0.696043,0.406143
9,10,0.844605,0.370202,0.174600,0.730198,0.529901


In [ ]:
# Add metadata and save
monthly_index["station_id"] = "HKRE0_Nairobi_-_Doonholm"
monthly_index["last_computed_at"] = pd.Timestamp.now()

monthly_index.to_csv(
    "HKRE0_Nairobi_-_Doonholm_monthly_climate_index.csv",
    index=False
)